# Check the size of EventKG dataset

In [1]:
from pathlib import Path

def human_size(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024

path = Path("F:\\EventKG\\data\\output_light")
for f in sorted(path.rglob("*")):
    if f.is_file():
        print(f"{human_size(f.stat().st_size):>12}  {f}")

     19.3 GB  F:\EventKG\data\output_light\entities.nt
      1.3 GB  F:\EventKG\data\output_light\events.nt
      9.4 MB  F:\EventKG\data\output_light\events_descriptions_from_text_events.nt
    307.6 MB  F:\EventKG\data\output_light\events_first_sentences.nt
      3.3 GB  F:\EventKG\data\output_light\preferred_labels.nt
      1.4 MB  F:\EventKG\data\output_light\property_labels.nt
     23.0 GB  F:\EventKG\data\output_light\relations_entities_other.nt
      9.9 GB  F:\EventKG\data\output_light\relations_entities_temporal.nt
      7.0 GB  F:\EventKG\data\output_light\relations_entity_base.nt
    796.8 MB  F:\EventKG\data\output_light\relations_event_base.nt
    628.3 MB  F:\EventKG\data\output_light\relations_events_literals.nt
      4.9 GB  F:\EventKG\data\output_light\relations_events_other.nt
      2.7 KB  F:\EventKG\data\output_light\schema.ttl
      1.3 GB  F:\EventKG\data\output_light\text_events.nt
    368.5 KB  F:\EventKG\data\output_light\type_labels_dbpedia.nt
      5.1 GB  F:

In [2]:
def peek_lines(path, n=10):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.strip())

print("=== relations_entities_temporal.nt ===")
peek_lines(r"F:\EventKG\data\output_light\relations_entities_temporal.nt", 10)

print("\n=== preferred_labels.nt ===")
peek_lines(r"F:\EventKG\data\output_light\preferred_labels.nt", 10)

print("\n=== property_labels.nt ===")
peek_lines(r"F:\EventKG\data\output_light\property_labels.nt", 10)

=== relations_entities_temporal.nt ===
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://eventkg.l3s.uni-hannover.de/schema/Relation> .
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> <https://eventkg.l3s.uni-hannover.de/resource/entity_2408355> .
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> <https://eventkg.l3s.uni-hannover.de/resource/entity_3723872> .
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <http://semanticweb.cs.vu.nl/2009/11/sem/roleType> <http://www.wikidata.org/prop/direct/P551> .
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <http://semanticweb.cs.vu.nl/2009/11/sem/hasBeginTimeStamp> "1920-01-01"^^<http://www.w3.org/2001/XMLSchema#date> .
<https://eventkg.l3s.uni-hannover.de/resource/relation_7> <https://eventkg.l3s.uni-hannover.de/schema/startUnitType> <ht

In [4]:
# Sanity check: confirm relation IDs are NOT interleaved
import re

NT_LINE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.+?)\s*\.\s*$')

seen_ids = []
with open(r"F:\EventKG\data\output_light\relations_entities_temporal.nt",
          "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 200:  # check first 200 lines only
            break
        m = NT_LINE.match(line.strip())
        if m:
            seen_ids.append(m.group(1))

# If contiguous, each ID should appear as a single unbroken run
from itertools import groupby
runs = [k for k, _ in groupby(seen_ids)]
print(f"Distinct relation IDs in first 200 lines: {len(set(seen_ids))}")
print(f"Number of contiguous runs: {len(runs)}")
print("✅ Contiguous!" if len(runs) == len(set(seen_ids)) else "⚠️ NOT contiguous — interleaved!")

Distinct relation IDs in first 200 lines: 29
Number of contiguous runs: 29
✅ Contiguous!


In [5]:
import re
from itertools import groupby

# ============================================================
# CONFIG — adjust paths as needed
# ============================================================
TEMPORAL_RELATIONS_PATH = r"F:\EventKG\data\output_light\relations_entities_temporal.nt"
LABELS_PATH              = r"F:\EventKG\data\output_light\preferred_labels.nt"
PROPERTY_LABELS_PATH     = r"F:\EventKG\data\output_light\property_labels.nt"

QUADRUPLES_OUT = r"F:\EventKG\data\eventkg_quadruples.tsv"
LABELS_OUT     = r"F:\EventKG\data\eventkg_labels.tsv"
PROP_LABELS_OUT = r"F:\EventKG\data\eventkg_property_labels.tsv"

TARGET_LANGS = {"en", "fr", "de", "ar", "fa"}

# ============================================================
# Sanity check: confirm relation IDs are contiguous
# (Run once before the full extraction — already confirmed OK,
#  kept here for reproducibility / re-verification if needed)
# ============================================================
def check_contiguity(path, n_lines=200):
    NT_LINE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.+?)\s*\.\s*$')
    seen_ids = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= n_lines:
                break
            m = NT_LINE.match(line.strip())
            if m:
                seen_ids.append(m.group(1))
    runs = [k for k, _ in groupby(seen_ids)]
    is_contiguous = len(runs) == len(set(seen_ids))
    print(f"Distinct relation IDs in first {n_lines} lines: {len(set(seen_ids))}")
    print(f"Number of contiguous runs: {len(runs)}")
    print("✅ Contiguous!" if is_contiguous else "⚠️ NOT contiguous — interleaved!")
    return is_contiguous


# ============================================================
# Step 1: Extract temporal quadruples from relations_entities_temporal.nt
# ============================================================
NT_LINE = re.compile(r'^<([^>]+)>\s+<([^>]+)>\s+(.+?)\s*\.\s*$')

P_SUBJECT  = "http://www.w3.org/1999/02/22-rdf-syntax-ns#subject"
P_OBJECT   = "http://www.w3.org/1999/02/22-rdf-syntax-ns#object"
P_ROLETYPE = "http://semanticweb.cs.vu.nl/2009/11/sem/roleType"
P_BEGIN    = "http://semanticweb.cs.vu.nl/2009/11/sem/hasBeginTimeStamp"
P_END      = "http://semanticweb.cs.vu.nl/2009/11/sem/hasEndTimeStamp"

def parse_object(o_raw):
    """Object is either <uri> or a literal "value"^^<type> / "value"@lang"""
    o_raw = o_raw.strip()
    if o_raw.startswith("<") and o_raw.endswith(">"):
        return o_raw[1:-1], "uri"
    m = re.match(r'^"([^"]*)"', o_raw)
    if m:
        return m.group(1), "literal"
    return o_raw, "literal"

def stream_temporal_quadruples(path, save_path, flush_every=50000):
    current_rel_id = None
    current = {}
    buffer = []
    count = 0
    line_no = 0

    with open(path, "r", encoding="utf-8", errors="ignore") as f_in, \
         open(save_path, "w", encoding="utf-8") as f_out:

        f_out.write("head\trelation\ttail\tbegin_time\tend_time\n")

        for line in f_in:
            line_no += 1
            m = NT_LINE.match(line.strip())
            if not m:
                continue
            s, p, o_raw = m.groups()

            if s != current_rel_id:
                # flush previous relation if complete enough
                if current.get("head") and current.get("tail") and current.get("rel"):
                    buffer.append(
                        f"{current['head']}\t{current['rel']}\t{current['tail']}\t"
                        f"{current.get('begin','')}\t{current.get('end','')}\n"
                    )
                    count += 1
                current_rel_id = s
                current = {}

            o_val, _ = parse_object(o_raw)

            if p == P_SUBJECT:
                current["head"] = o_val
            elif p == P_OBJECT:
                current["tail"] = o_val
            elif p == P_ROLETYPE:
                current["rel"] = o_val
            elif p == P_BEGIN:
                current["begin"] = o_val
            elif p == P_END:
                current["end"] = o_val

            if len(buffer) >= flush_every:
                f_out.writelines(buffer)
                buffer = []
                print(f"  line {line_no:,} | {count:,} quadruples extracted so far...", end="\r")

        # flush the very last relation + remaining buffer
        if current.get("head") and current.get("tail") and current.get("rel"):
            buffer.append(
                f"{current['head']}\t{current['rel']}\t{current['tail']}\t"
                f"{current.get('begin','')}\t{current.get('end','')}\n"
            )
            count += 1
        if buffer:
            f_out.writelines(buffer)

    print(f"\nDone. {count:,} temporal quadruples extracted -> {save_path}")
    return count


# ============================================================
# Step 2: Extract multilingual entity labels from preferred_labels.nt
# ============================================================
LABEL_PATTERN = re.compile(r'^<([^>]+)>\s+<[^>]+>\s+"(.*)"@(\w+)\s*\.\s*$')

def extract_labels(path, save_path, langs=TARGET_LANGS):
    count = 0
    line_no = 0
    with open(path, "r", encoding="utf-8", errors="ignore") as f_in, \
         open(save_path, "w", encoding="utf-8") as f_out:
        f_out.write("entity\tlabel\tlang\n")
        for line in f_in:
            line_no += 1
            m = LABEL_PATTERN.match(line.strip())
            if m:
                entity, label, lang = m.groups()
                if lang in langs:
                    label_clean = label.replace("\t", " ")
                    f_out.write(f"{entity}\t{label_clean}\t{lang}\n")
                    count += 1
            if line_no % 200000 == 0:
                print(f"  line {line_no:,} | {count:,} labels extracted so far...", end="\r")
    print(f"\nDone. {count:,} labels extracted -> {save_path}")
    return count


# ============================================================
# Step 3: Extract multilingual relation/property labels
# ============================================================
def extract_property_labels(path, save_path, langs=TARGET_LANGS):
    count = 0
    line_no = 0
    pattern = re.compile(r'^<([^>]+)>\s+<[^>]+>\s+"(.*)"@(\w+)\s*\.\s*$')
    with open(path, "r", encoding="utf-8", errors="ignore") as f_in, \
         open(save_path, "w", encoding="utf-8") as f_out:
        f_out.write("property\tlabel\tlang\n")
        for line in f_in:
            line_no += 1
            m = pattern.match(line.strip())
            if m:
                prop, label, lang = m.groups()
                if lang in langs:
                    label_clean = label.replace("\t", " ")
                    f_out.write(f"{prop}\t{label_clean}\t{lang}\n")
                    count += 1
            if line_no % 200000 == 0:
                print(f"  line {line_no:,} | {count:,} property labels extracted...", end="\r")
    print(f"\nDone. {count:,} property labels extracted -> {save_path}")
    return count


# ============================================================
# RUN EVERYTHING
# ============================================================
if __name__ == "__main__":
    print("=== Step 0: Contiguity check ===")
    check_contiguity(TEMPORAL_RELATIONS_PATH)

    print("\n=== Step 1: Extracting temporal quadruples (9.9 GB — will take a while) ===")
    stream_temporal_quadruples(TEMPORAL_RELATIONS_PATH, QUADRUPLES_OUT)

    print("\n=== Step 2: Extracting multilingual entity labels (3.3 GB) ===")
    extract_labels(LABELS_PATH, LABELS_OUT)

    print("\n=== Step 3: Extracting multilingual property labels (1.4 MB — fast) ===")
    extract_property_labels(PROPERTY_LABELS_PATH, PROP_LABELS_OUT)

    print("\n✅ All extractions complete.")
    print(f"  Quadruples:       {QUADRUPLES_OUT}")
    print(f"  Entity labels:    {LABELS_OUT}")
    print(f"  Property labels:  {PROP_LABELS_OUT}")

=== Step 0: Contiguity check ===
Distinct relation IDs in first 200 lines: 29
Number of contiguous runs: 29
✅ Contiguous!

=== Step 1: Extracting temporal quadruples (9.9 GB — will take a while) ===
  line 60,963,365 | 8,600,000 quadruples extracted so far...
Done. 8,649,760 temporal quadruples extracted -> F:\EventKG\data\eventkg_quadruples.tsv

=== Step 2: Extracting multilingual entity labels (3.3 GB) ===
  line 24,600,000 | 22,972,267 labels extracted so far...
Done. 23,044,928 labels extracted -> F:\EventKG\data\eventkg_labels.tsv

=== Step 3: Extracting multilingual property labels (1.4 MB — fast) ===

Done. 3,736 property labels extracted -> F:\EventKG\data\eventkg_property_labels.tsv

✅ All extractions complete.
  Quadruples:       F:\EventKG\data\eventkg_quadruples.tsv
  Entity labels:    F:\EventKG\data\eventkg_labels.tsv
  Property labels:  F:\EventKG\data\eventkg_property_labels.tsv


In [6]:
import csv

with open(r"F:\EventKG\data\eventkg_quadruples.tsv") as f:
    reader = csv.reader(f, delimiter="\t")
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(row)

['head', 'relation', 'tail', 'begin_time', 'end_time']
['https://eventkg.l3s.uni-hannover.de/resource/entity_2408355', 'http://www.wikidata.org/prop/direct/P551', 'https://eventkg.l3s.uni-hannover.de/resource/entity_3723872', '1920-01-01', '1938-12-31']
['https://eventkg.l3s.uni-hannover.de/resource/entity_15371095', 'http://www.wikidata.org/prop/direct/P54', 'https://eventkg.l3s.uni-hannover.de/resource/entity_1917938', '2005-01-01', '2006-06-30']
['https://eventkg.l3s.uni-hannover.de/resource/entity_9239556', 'http://www.wikidata.org/prop/direct/P54', 'https://eventkg.l3s.uni-hannover.de/resource/entity_11582653', '1962-01-01', '1967-12-31']
['https://eventkg.l3s.uni-hannover.de/resource/entity_8394950', 'http://www.wikidata.org/prop/direct/P1435', 'https://eventkg.l3s.uni-hannover.de/resource/entity_7232634', '1962-10-17', '']


In [7]:
with open(r"F:\EventKG\data\eventkg_labels.tsv") as f:
    reader = csv.reader(f, delimiter="\t")
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(row)

['entity', 'label', 'lang']
['https://eventkg.l3s.uni-hannover.de/resource/event_127959', 'Battle of Tulagi and Gavutuâ€“Tanambogo', 'en']
['https://eventkg.l3s.uni-hannover.de/resource/event_395347', 'South Australian state election in Black, 2018', 'en']
['https://eventkg.l3s.uni-hannover.de/resource/event_2624978', 'Ech Du', 'en']
['https://eventkg.l3s.uni-hannover.de/resource/event_2888615', '2021 African Minifootball Cup', 'en']


In [2]:
import csv

with open(r"F:\EventKG\data\eventkg_property_labels.tsv") as f:
    reader = csv.reader(f, delimiter="\t")
    for i, row in enumerate(reader):
        if i >= 5:
            break
        print(row)

['property', 'label', 'lang']
['http://www.wikidata.org/prop/direct/P1479', 'hat mitwirkenden Faktor', 'de']
['http://www.wikidata.org/prop/direct/P6684', 'premiÃ¨re attestation dans', 'fr']
['http://www.wikidata.org/prop/direct/P5206', 'a la classe de conjugaison', 'fr']
['http://www.wikidata.org/prop/direct/P4238', 'webcam page URL', 'en']


In [1]:
# Check if ANY provenance/source info survived in the temporal relations file
# by looking for predicates we haven't seen yet
import re

predicates_seen = set()
with open(r"F:\EventKG\data\output_light\relations_entities_temporal.nt",
          "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i > 500000:  # sample first 500k lines, not the whole 9.9GB
            break
        m = re.match(r'^<[^>]+>\s+<([^>]+)>\s+', line.strip())
        if m:
            predicates_seen.add(m.group(1))

for p in sorted(predicates_seen):
    print(p)

http://semanticweb.cs.vu.nl/2009/11/sem/hasBeginTimeStamp
http://semanticweb.cs.vu.nl/2009/11/sem/hasEndTimeStamp
http://semanticweb.cs.vu.nl/2009/11/sem/roleType
http://www.w3.org/1999/02/22-rdf-syntax-ns#object
http://www.w3.org/1999/02/22-rdf-syntax-ns#subject
http://www.w3.org/1999/02/22-rdf-syntax-ns#type
https://eventkg.l3s.uni-hannover.de/schema/endUnitType
https://eventkg.l3s.uni-hannover.de/schema/startUnitType


# Check label coverage per language

In [3]:
import pandas as pd

LABELS_PATH = r"F:\EventKG\data\eventkg_labels.tsv"
QUADRUPLES_PATH = r"F:\EventKG\data\eventkg_quadruples.tsv"

# Load labels
df_labels = pd.read_csv(LABELS_PATH, sep="\t")
print(f"Total label rows: {len(df_labels):,}")
print(f"\nLabel count per language:")
print(df_labels['lang'].value_counts())

print(f"\nUnique entities with at least one label: {df_labels['entity'].nunique():,}")

Total label rows: 23,044,928

Label count per language:
lang
en    21580417
fr      752899
de      711612
Name: count, dtype: int64

Unique entities with at least one label: 23,044,928


# check pairwise overlaps

In [1]:
import pandas as pd

LABELS_PATH = r"F:\EventKG\data\eventkg_labels.tsv"
QUADRUPLES_PATH = r"F:\EventKG\data\eventkg_quadruples.tsv"

# Load labels
df_labels = pd.read_csv(LABELS_PATH, sep="\t")

# Build entity -> set of languages it has a label in
entity_langs = df_labels.groupby('entity')['lang'].apply(set).to_dict()

print(f"Total entities with labels: {len(entity_langs):,}\n")

# Check pairwise overlap for all language combinations present
langs_present = df_labels['lang'].unique()
print("Pairwise entity overlap (entities with labels in BOTH languages):")
print(f"{'Lang A':6s} {'Lang B':6s} {'A only':>10s} {'B only':>10s} {'Both (A∩B)':>12s}")

from itertools import combinations
for lang_a, lang_b in combinations(sorted(langs_present), 2):
    ents_a = {e for e, langs in entity_langs.items() if lang_a in langs}
    ents_b = {e for e, langs in entity_langs.items() if lang_b in langs}
    both = ents_a & ents_b
    print(f"{lang_a:6s} {lang_b:6s} {len(ents_a - ents_b):>10,} {len(ents_b - ents_a):>10,} {len(both):>12,}")

Total entities with labels: 23,044,928

Pairwise entity overlap (entities with labels in BOTH languages):
Lang A Lang B     A only     B only   Both (A∩B)
de     en        711,612 21,580,417            0
de     fr        711,612    752,899            0
en     fr     21,580,417    752,899            0


In [1]:
# Check the RAW file directly - does any entity appear more than once?
from collections import Counter

entity_counts = Counter()
with open(r"F:\EventKG\data\output_light\preferred_labels.nt", "r", encoding="utf-8", errors="ignore") as f:
    for i, line in enumerate(f):
        if i >= 2_000_000:  # sample first 2M lines
            break
        import re
        m = re.match(r'^<([^>]+)>', line.strip())
        if m:
            entity_counts[m.group(1)] += 1

# How many entities appear more than once in this sample?
multi = {e: c for e, c in entity_counts.items() if c > 1}
print(f"Entities appearing more than once (first 2M lines): {len(multi)}")
print(f"Sample of repeated entities:")
for e, c in list(multi.items())[:10]:
    print(f"  {e}: {c} times")

Entities appearing more than once (first 2M lines): 0
Sample of repeated entities:


In [2]:
def peek_lines(path, n=20):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.strip())

print("=== entities.nt ===")
peek_lines(r"F:\EventKG\data\output_light\entities.nt", 20)

=== entities.nt ===
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://semanticweb.cs.vu.nl/2009/11/sem/Actor> .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2002/07/owl#sameAs> <http://www.wikidata.org/entity/Q52679700> .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2000/01/rdf-schema#label> "Maria D. Setzu"@en .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2000/01/rdf-schema#label> "Maria D. Setzu"@it .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2000/01/rdf-schema#label> "Maria D Setzu"@nl .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2000/01/rdf-schema#label> "Maria D. Setzu"@es .
<https://eventkg.l3s.uni-hannover.de/resource/entity_13901148> <http://www.w3.org/2000/01/rdf-schema#label> "Maria D Setzu"@sl .
<https://eventkg.l3s.u

In [1]:
import re

ENTITIES_PATH = r"F:\EventKG\data\output_light\entities.nt"
LABELS_OUT    = r"F:\EventKG\data\eventkg_entity_labels_v2.tsv"
SAMEAS_OUT    = r"F:\EventKG\data\eventkg_sameas_v2.tsv"

TARGET_LANGS = {"en", "fr", "de", "ar", "fa"}

LABEL_PATTERN  = re.compile(r'^<([^>]+)>\s+<http://www\.w3\.org/2000/01/rdf-schema#label>\s+"(.*)"@(\w+)\s*\.\s*$')
SAMEAS_PATTERN = re.compile(r'^<([^>]+)>\s+<http://www\.w3\.org/2002/07/owl#sameAs>\s+<([^>]+)>\s*\.\s*$')

def extract_entities_file(path, labels_out, sameas_out, langs=TARGET_LANGS):
    label_count = 0
    sameas_count = 0
    line_no = 0

    with open(path, "r", encoding="utf-8", errors="ignore") as f_in, \
         open(labels_out, "w", encoding="utf-8") as f_labels, \
         open(sameas_out, "w", encoding="utf-8") as f_sameas:

        f_labels.write("entity\tlabel\tlang\n")
        f_sameas.write("entity\ttarget_uri\n")

        for line in f_in:
            line_no += 1
            line_s = line.strip()

            m_label = LABEL_PATTERN.match(line_s)
            if m_label:
                entity, label, lang = m_label.groups()
                if lang in langs:
                    label_clean = label.replace("\t", " ")
                    f_labels.write(f"{entity}\t{label_clean}\t{lang}\n")
                    label_count += 1
                continue

            m_sameas = SAMEAS_PATTERN.match(line_s)
            if m_sameas:
                entity, target = m_sameas.groups()
                f_sameas.write(f"{entity}\t{target}\n")
                sameas_count += 1
                continue

            if line_no % 1_000_000 == 0:
                print(f"  line {line_no:,} | labels: {label_count:,} | sameAs: {sameas_count:,}", end="\r")

    print(f"\nDone. {label_count:,} labels, {sameas_count:,} sameAs links extracted.")
    return label_count, sameas_count

extract_entities_file(ENTITIES_PATH, LABELS_OUT, SAMEAS_OUT)

  line 115,000,000 | labels: 31,286,788 | sameAs: 33,422,440
Done. 36,991,589 labels, 44,414,699 sameAs links extracted.


(36991589, 44414699)

In [2]:
import pandas as pd

LABELS_OUT = r"F:\EventKG\data\eventkg_entity_labels_v2.tsv"

df_labels = pd.read_csv(LABELS_OUT, sep="\t")
print(f"Total label rows: {len(df_labels):,}")
print(f"\nLabel count per language:")
print(df_labels['lang'].value_counts())

entity_langs = df_labels.groupby('entity')['lang'].apply(set).to_dict()
print(f"\nUnique entities with at least one (target-language) label: {len(entity_langs):,}")

# Check: do entities now actually have multiple languages?
multi_lang = {e: langs for e, langs in entity_langs.items() if len(langs) > 1}
print(f"Entities with 2+ target-language labels: {len(multi_lang):,}")

from itertools import combinations
langs_present = df_labels['lang'].unique()
print("\nPairwise entity overlap:")
print(f"{'Lang A':6s} {'Lang B':6s} {'A only':>10s} {'B only':>10s} {'Both (A∩B)':>12s}")
for lang_a, lang_b in combinations(sorted(langs_present), 2):
    ents_a = {e for e, langs in entity_langs.items() if lang_a in langs}
    ents_b = {e for e, langs in entity_langs.items() if lang_b in langs}
    both = ents_a & ents_b
    print(f"{lang_a:6s} {lang_b:6s} {len(ents_a - ents_b):>10,} {len(ents_b - ents_a):>10,} {len(both):>12,}")

Total label rows: 36,991,589

Label count per language:
lang
en    21903746
fr     7887448
de     7200395
Name: count, dtype: int64

Unique entities with at least one (target-language) label: 22,172,306
Entities with 2+ target-language labels: 8,589,288

Pairwise entity overlap:
Lang A Lang B     A only     B only   Both (A∩B)
de     en        678,047 14,608,106    6,178,077
de     fr      2,578,019  3,082,276    4,278,105
en     fr     14,170,889    745,087    6,615,294


# Build EN-FR cross-lingual TKG

In [1]:
import pandas as pd
import os

QUADRUPLES_PATH = r"F:\EventKG\data\eventkg_quadruples.tsv"
LABELS_PATH     = r"F:\EventKG\data\eventkg_entity_labels_v2.tsv"

LANG_A = "en"
LANG_B = "fr"
OUT_DIR = rf"F:\EventKG\data\crosslingual_{LANG_A}_{LANG_B}"
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading labels...")
df_labels = pd.read_csv(LABELS_PATH, sep="\t")
entity_langs = df_labels.groupby('entity')['lang'].apply(set).to_dict()

print("Loading quadruples...")
df_quad = pd.read_csv(QUADRUPLES_PATH, sep="\t", dtype=str)
print(f"Total quadruples: {len(df_quad):,}")

ents_a = {e for e, langs in entity_langs.items() if LANG_A in langs}
ents_b = {e for e, langs in entity_langs.items() if LANG_B in langs}
shared_candidates = ents_a & ents_b

print(f"\nEntities with {LANG_A} label: {len(ents_a):,}")
print(f"Entities with {LANG_B} label: {len(ents_b):,}")
print(f"Shared (alignment candidates): {len(shared_candidates):,}")

# Filter quadruples: BOTH head and tail must have a label in that language
mask_a = df_quad['head'].isin(ents_a) & df_quad['tail'].isin(ents_a)
mask_b = df_quad['head'].isin(ents_b) & df_quad['tail'].isin(ents_b)

df_a = df_quad[mask_a].copy()
df_b = df_quad[mask_b].copy()

print(f"\n{LANG_A} KG quadruples: {len(df_a):,}")
print(f"{LANG_B} KG quadruples: {len(df_b):,}")

present_a = set(df_a['head']) | set(df_a['tail'])
present_b = set(df_b['head']) | set(df_b['tail'])
real_alignment = present_a & present_b

print(f"\nEntities actually present in {LANG_A} KG: {len(present_a):,}")
print(f"Entities actually present in {LANG_B} KG: {len(present_b):,}")
print(f"Real alignment pairs (present in both, post-filtering): {len(real_alignment):,}")

# Save
df_a.to_csv(f"{OUT_DIR}/quadruples_{LANG_A}.tsv", sep="\t", index=False)
df_b.to_csv(f"{OUT_DIR}/quadruples_{LANG_B}.tsv", sep="\t", index=False)
pd.DataFrame({"entity": sorted(real_alignment)}).to_csv(
    f"{OUT_DIR}/alignment_pairs.tsv", sep="\t", index=False)

print(f"\nSaved to {OUT_DIR}")

Loading labels...
Loading quadruples...
Total quadruples: 8,649,760

Entities with en label: 20,786,183
Entities with fr label: 7,360,381
Shared (alignment candidates): 6,615,294

en KG quadruples: 7,932,967
fr KG quadruples: 4,088,530

Entities actually present in en KG: 4,245,833
Entities actually present in fr KG: 1,696,492
Real alignment pairs (present in both, post-filtering): 1,460,771

Saved to F:\EventKG\data\crosslingual_en_fr
